# 🛡️ FakeGuard — AI Fake News Detector
## NeuroLogic '26 Datathon Submission

**Model:** RoBERTa (`roberta-base`) fine-tuned on WELFake Dataset  
**Task:** Binary classification — REAL (1) vs FAKE (0)  
**Target:** Beat TF-IDF baseline (~94.66%) → achieve ~97-98%+ accuracy  

### Pipeline Overview
```
Cell 1 → Environment setup + GPU check
Cell 2 → Clone repo + configure paths
Cell 3 → Data preprocessing
Cell 4 → Baseline model (TF-IDF + Logistic Regression)
Cell 5 → RoBERTa fine-tuning (~20-30 min)
Cell 6 → Evaluation + confusion matrix + error analysis
Cell 7 → Generate submission predictions.csv
```

## Cell 1 — Environment Setup & GPU Check

In [ ]:
# ============================================================
# CELL 1 — Environment Setup & GPU Check
# ============================================================
# Kaggle has most packages pre-installed.
# We only need to install/upgrade transformers, accelerate, seaborn.
# torch + sklearn + pandas + numpy are pre-installed on Kaggle kernels.
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('Installing / upgrading required packages...')
install('transformers==4.40.0')
install('accelerate==0.29.0')
install('seaborn==0.13.2')
install('datasets==2.19.0')
print('✅ Packages ready.')

# --- GPU Check ---
import torch
print(f'\nPyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU device      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU detected! Go to Settings → Accelerator → GPU (P100)')
    print('   Training will take 4-8 hours on CPU — do not proceed without GPU.')
    raise RuntimeError('GPU required. Enable it in Kaggle notebook settings.')

import os, random, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
print('\n✅ All imports successful.')

# Reproducibility — set seed everywhere
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print(f'✅ Random seed set to {SEED}')

## Cell 2 — Clone Repository & Configure Paths

In [ ]:
# ============================================================
# CELL 2 — Clone GitHub Repo & Configure Paths
# ============================================================
# This clones your repo so all src/ modules are importable.
# The dataset is loaded from Kaggle's /kaggle/input/ directory.

import subprocess, os, sys

REPO_URL  = 'https://github.com/ayushtiwari18/neurologic-datathon-fakenews.git'
REPO_NAME = 'neurologic-datathon-fakenews'

# Clone repo (skip if already cloned)
if not os.path.exists(f'/kaggle/working/{REPO_NAME}'):
    print('Cloning repository...')
    subprocess.check_call(['git', 'clone', REPO_URL, f'/kaggle/working/{REPO_NAME}'])
    print('✅ Repository cloned.')
else:
    print('✅ Repository already exists — pulling latest changes...')
    subprocess.check_call(['git', '-C', f'/kaggle/working/{REPO_NAME}', 'pull'])

# Add repo root to Python path so src/ imports work
REPO_ROOT = f'/kaggle/working/{REPO_NAME}'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
print(f'Working directory: {os.getcwd()}')

# ── Paths ────────────────────────────────────────────────
# Kaggle datasets are mounted at /kaggle/input/<dataset-name>/
# Upload WELFake_Dataset.csv as a Kaggle dataset named 'welfake-dataset'
# OR point DATASET_PATH to wherever you uploaded it.

import glob

# Auto-detect WELFake CSV in /kaggle/input/
welfake_candidates = glob.glob('/kaggle/input/**/WELFake_Dataset.csv', recursive=True)

if welfake_candidates:
    DATASET_PATH = welfake_candidates[0]
    print(f'✅ Auto-detected dataset: {DATASET_PATH}')
else:
    # Fallback: check if already processed
    DATASET_PATH = None
    print('⚠️  WELFake_Dataset.csv not found in /kaggle/input/')
    print('   Please add it as a Kaggle dataset (see instructions below):')
    print('   1. Go to: https://www.kaggle.com/datasets')
    print('   2. Click New Dataset → Upload WELFake_Dataset.csv')
    print('   3. In this notebook: Add Data → search your dataset name → Add')

# Output directories
os.makedirs(f'{REPO_ROOT}/data/raw', exist_ok=True)
os.makedirs(f'{REPO_ROOT}/data/processed', exist_ok=True)
os.makedirs(f'{REPO_ROOT}/outputs', exist_ok=True)
os.makedirs(f'{REPO_ROOT}/models/roberta_fakenews', exist_ok=True)
print('✅ Output directories created.')

## Cell 3 — Data Preprocessing

In [ ]:
# ============================================================
# CELL 3 — Data Preprocessing
# ============================================================
# This runs src/preprocess.py logic on the WELFake dataset.
# Produces: data/processed/train.csv, val.csv, test.csv
# Split: 70% train | 15% val | 15% test (stratified)

import shutil
from pathlib import Path

PROCESSED_DIR = Path(f'{REPO_ROOT}/data/processed')
train_csv = PROCESSED_DIR / 'train.csv'
val_csv   = PROCESSED_DIR / 'val.csv'
test_csv  = PROCESSED_DIR / 'test.csv'

if train_csv.exists() and val_csv.exists() and test_csv.exists():
    print('✅ Processed CSVs already exist — skipping preprocessing.')
    train_df = pd.read_csv(train_csv)
    val_df   = pd.read_csv(val_csv)
    test_df  = pd.read_csv(test_csv)
else:
    if DATASET_PATH is None:
        raise FileNotFoundError('WELFake_Dataset.csv not found. Add it as a Kaggle dataset first.')
    # Copy raw dataset to expected location
    raw_dest = Path(f'{REPO_ROOT}/data/raw/WELFake_Dataset.csv')
    shutil.copy2(DATASET_PATH, raw_dest)
    print(f'Dataset copied to {raw_dest}')
    # Run preprocessing pipeline
    from src.preprocess import run_preprocessing
    train_df, val_df, test_df = run_preprocessing()
    print('✅ Preprocessing complete.')

print(f'\nDataset split summary:')
print(f'  Train : {len(train_df):>7,} rows')
print(f'  Val   : {len(val_df):>7,} rows')
print(f'  Test  : {len(test_df):>7,} rows')
print(f'  Total : {len(train_df)+len(val_df)+len(test_df):>7,} rows')

# Label distribution check
print(f'\nTrain label distribution:')
print(train_df['label'].value_counts().rename({0: 'FAKE', 1: 'REAL'}).to_string())

# Quick EDA plots
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class distribution
label_counts = train_df['label'].map({0: 'FAKE', 1: 'REAL'}).value_counts()
axes[0].bar(label_counts.index, label_counts.values, color=['#F44336', '#2196F3'])
axes[0].set_title('Class Distribution (Train)', fontsize=13)
axes[0].set_ylabel('Count')

# Text length distribution
train_df['combined'].str.len().hist(bins=50, ax=axes[1], color='#4CAF50', edgecolor='white')
axes[1].set_title('Combined Text Length Distribution', fontsize=13)
axes[1].set_xlabel('Character count')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig(f'{REPO_ROOT}/outputs/eda_plots.png', dpi=150)
plt.show()
print('✅ EDA plots saved to outputs/eda_plots.png')

## Cell 4 — Baseline Model (TF-IDF + Logistic Regression)

In [ ]:
# ============================================================
# CELL 4 — Baseline Model (TF-IDF + Logistic Regression)
# ============================================================
# This takes ~2 minutes. It gives us our comparison baseline.
# Expected accuracy: ~94-95% on WELFake
# We already confirmed 94.66% locally — this verifies Kaggle gives same result.

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)

print('Building TF-IDF features...')
# IMPORTANT: fit ONLY on train — never on val (data leakage)
vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1, 2))
X_train = vectorizer.fit_transform(train_df['combined'])
X_val   = vectorizer.transform(val_df['combined'])
print(f'TF-IDF matrix: train={X_train.shape}, val={X_val.shape}')

print('Training Logistic Regression...')
lr_model = LogisticRegression(max_iter=1000, random_state=SEED)
lr_model.fit(X_train, train_df['label'])

baseline_preds = lr_model.predict(X_val)
baseline_acc   = accuracy_score(val_df['label'], baseline_preds)
baseline_f1    = f1_score(val_df['label'], baseline_preds, average='weighted')

print(f'\n{'='*50}')
print(f'  BASELINE RESULTS')
print(f'{'='*50}')
print(f'  Accuracy : {baseline_acc:.4f}')
print(f'  F1 (wt)  : {baseline_f1:.4f}')
print(f'{'='*50}')

# Save baseline confusion matrix
cm_base = confusion_matrix(val_df['label'], baseline_preds)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_base, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['FAKE','REAL'], yticklabels=['FAKE','REAL'], ax=ax)
ax.set_title(f'Baseline Confusion Matrix (Acc={baseline_acc:.4f})', fontsize=13)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout()
plt.savefig(f'{REPO_ROOT}/outputs/baseline_confusion_matrix.png', dpi=150)
plt.show()
print('✅ Baseline complete. Confusion matrix saved.')
print(classification_report(val_df['label'], baseline_preds, target_names=['FAKE','REAL']))

## Cell 5 — RoBERTa Fine-Tuning (~20-30 min on P100)

In [ ]:
# ============================================================
# CELL 5 — RoBERTa Fine-Tuning
# ============================================================
# Expected duration: ~20-30 min on Kaggle P100
# Expected val accuracy: ~97-98%+
#
# If you get CUDA OOM error:
#   → Change BATCH_SIZE below from 16 to 8 and re-run
#
# Checkpoints are saved every epoch to models/roberta_fakenews/
# so if the session dies, the last epoch is safe.

from src.train import run_training

print('Starting RoBERTa fine-tuning...')
print('Expected time: ~20-30 minutes on Kaggle P100')
print('Checkpoints saved every epoch — safe against session interruption')
print('='*60)

trainer, train_metrics = run_training(
    train_path = str(PROCESSED_DIR / 'train.csv'),
    val_path   = str(PROCESSED_DIR / 'val.csv'),
)

print('\n' + '='*60)
print('  TRAINING COMPLETE')
print('='*60)
print(f"  Val Accuracy : {train_metrics['final_val_accuracy']:.4f}")
print(f"  Val F1       : {train_metrics['final_val_f1']:.4f}")
print(f"  Train time   : {train_metrics['train_runtime_seconds']:.0f} seconds")
print('='*60)

## Cell 6 — Evaluation + Confusion Matrix + Error Analysis

In [ ]:
# ============================================================
# CELL 6 — Full Evaluation
# ============================================================
# Produces:
#   outputs/confusion_matrix.png     ← screenshot this for README
#   outputs/evaluation_report.json  ← full metrics JSON
#   outputs/error_analysis.csv      ← top 20 wrong predictions
#   Console comparison table        ← Baseline vs RoBERTa

from src.evaluate import run_evaluation

print('Running full evaluation...')

eval_report = run_evaluation(
    val_csv_path      = str(PROCESSED_DIR / 'val.csv'),
    baseline_accuracy = baseline_acc,   # use the exact baseline we computed above
)

# Display confusion matrix inline
from IPython.display import Image, display
cm_path = f'{REPO_ROOT}/outputs/confusion_matrix.png'
if os.path.exists(cm_path):
    display(Image(cm_path))

# Show top 10 error analysis examples
import pandas as pd
error_path = f'{REPO_ROOT}/outputs/error_analysis.csv'
if os.path.exists(error_path):
    error_df = pd.read_csv(error_path)
    print(f'\nTop 10 Most Confident Wrong Predictions:')
    pd.set_option('display.max_colwidth', 80)
    display(error_df[['true_label','predicted_label','confidence','uncertain_flag']].head(10))

roberta_acc = eval_report['metrics']['accuracy']
print(f'\n  Improvement over baseline: +{roberta_acc - baseline_acc:.4f} ({(roberta_acc-baseline_acc)*100:.2f}%)')

## Cell 7 — Generate Submission Predictions

In [ ]:
# ============================================================
# CELL 7 — Generate Competition Submission Predictions
# ============================================================
# Produces: outputs/predictions.csv
#
# IMPORTANT (from instructions/08_key_failure_points.md):
#   If competition test.csv has NO label column, this script handles
#   it gracefully — it will skip metric computation and output
#   predictions only. Never try to compute accuracy on unlabeled test data.

from src.predict import run_predict

print('Generating predictions on test set...')

predictions_df = run_predict(
    test_csv_path = str(PROCESSED_DIR / 'test.csv'),
)

# Show sample predictions
print('\nSample predictions (first 10 rows):')
display(predictions_df[['id','label','confidence','uncertain']].head(10))

# Final output files summary
print('\n' + '='*60)
print('  ALL OUTPUT FILES')
print('='*60)
output_files = [
    ('outputs/predictions.csv',            'Competition submission file'),
    ('outputs/confusion_matrix.png',        'Confusion matrix (screenshot for README)'),
    ('outputs/baseline_confusion_matrix.png','Baseline confusion matrix'),
    ('outputs/evaluation_report.json',      'Full metrics report'),
    ('outputs/error_analysis.csv',          'Top 20 misclassified examples'),
    ('outputs/training_metrics.json',       'Training loss + accuracy history'),
    ('outputs/eda_plots.png',               'EDA class + text length plots'),
]
for fpath, desc in output_files:
    full = f'{REPO_ROOT}/{fpath}'
    status = '✅' if os.path.exists(full) else '❌ MISSING'
    print(f'  {status}  {fpath:<45} {desc}')

print('\n' + '='*60)
print('  COMPETITION SUMMARY')
print('='*60)
print(f"  Baseline Accuracy  : {baseline_acc:.4f}")
print(f"  RoBERTa Accuracy   : {eval_report['metrics']['accuracy']:.4f}")
print(f"  Improvement        : +{eval_report['metrics']['accuracy'] - baseline_acc:.4f}")
print(f"  F1 (weighted)      : {eval_report['metrics']['f1_weighted']:.4f}")
print(f"  Val samples        : {eval_report['val_samples']:,}")
print('='*60)
print('\n✅ Notebook complete! Download outputs/predictions.csv for submission.')